# Xây dựng mô hình Naïve ngây thơ trên tập dữ liệu mushroom. Dữ liệu lấy tại https://www.kaggle.com/datasets/uciml/mushroom-classification/data

## 1. Mục tiêu

- Làm quen với thuật toán Naïve Bayes trong bài toán phân loại.
- Áp dụng mô hình để dự đoán nấm độc hay nấm ăn được.
- Tiền xử lý dữ liệu có nhiều thuộc tính dạng chuỗi phân loại (categorical).
- Đánh giá mô hình bằng các chỉ số:
  + Accuracy
  + Confusion Matrix
  + Precision – Recall – F1-score.



## 2. Import thư viện và nạp dữ liệu

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Đọc dữ liệu nấm
df = pd.read_csv("mushrooms.csv")

print("5 dòng đầu tiên của dữ liệu:")
print(df.head())

print("Thông tin dữ liệu:")
print(df.info())

5 dòng đầu tiên của dữ liệu:
  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                      w    

**Nhận xét dữ liệu ban đầu:**
- Dữ liệu gồm 8124 dòng và 23 cột.
- Tất cả các cột đều có kiểu object (categorical).
- Không có giá trị bị thiếu (non-null = 8124 ở tất cả các cột).
- Cột mục tiêu là “class”, gồm 2 giá trị:
  + e = edible (ăn được)
  + p = poisonous (độc)
- Tất cả thuộc tính đều ở dạng ký tự (a, b, c, …) → cần mã hóa để mô hình xử lý.


## 3. Tiền xử lý dữ liệu

In [2]:
# (a) Mã hóa toàn bộ dữ liệu dạng chuỗi
data = df.copy()

label_encoder = LabelEncoder()

for col in data.columns:
    data[col] = label_encoder.fit_transform(data[col])

# (b) Tách đặc trưng (x) và nhãn (y)
x = data.drop("class", axis=1)
y = data["class"]

# (c) Chia dữ liệu train/test (80% train – 20% test)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [3]:
print("Dữ liệu sau tiền xử lý (5 dòng đầu):")
print(data.head())

Dữ liệu sau tiền xử lý (5 dòng đầu):
   class  cap-shape  cap-surface  cap-color  bruises  odor  gill-attachment  \
0      1          5            2          4        1     6                1   
1      0          5            2          9        1     0                1   
2      0          0            2          8        1     3                1   
3      1          5            3          8        1     6                1   
4      0          5            2          3        0     5                1   

   gill-spacing  gill-size  gill-color  ...  stalk-surface-below-ring  \
0             0          1           4  ...                         2   
1             0          0           4  ...                         2   
2             0          0           5  ...                         2   
3             0          1           5  ...                         2   
4             1          0           4  ...                         2   

   stalk-color-above-ring  stalk-color-below-ring

**Nhận xét về tiền xử lý:**
- Toàn bộ 23 cột đã được chuyển về dạng số.
- Vì tất cả các cột đều là dữ liệu phân loại → không cần chuẩn hóa (StandardScaler).
- Tập train/test 80/20 là phù hợp với kích thước dữ liệu lớn (8124 mẫu).
- Dữ liệu sau xử lý hoàn toàn phù hợp để đưa vào mô hình Naïve Bayes.

## 4. Xây dựng mô hình Naïve Bayes

Dữ liệu nấm là phân loại rời rạc → sử dụng:

In [4]:
model = MultinomialNB()
model.fit(x_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


## 5. Đánh giá mô hình

In [5]:
# Dự đoán trên tập test
y_pred = model.predict(x_test)

# Tính các chỉ số đánh giá
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classif_report = classification_report(y_test, y_pred)

print(f"\nAccuracy", accuracy)
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classif_report)


Accuracy 0.8073846153846154

Confusion Matrix:
[[792  51]
 [262 520]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.94      0.84       843
           1       0.91      0.66      0.77       782

    accuracy                           0.81      1625
   macro avg       0.83      0.80      0.80      1625
weighted avg       0.83      0.81      0.80      1625



**Nhận xét chi tiết về kết quả:**

**1. Độ chính xác (Accuracy)**
- Độ chính xác của mô hình chỉ đạt ~80.7%, khá thấp so với kỳ vọng của bộ dữ liệu mushroom (thường đạt 95–100%). Điều này cho thấy mô hình chưa khai thác tốt mối quan hệ giữa các thuộc tính trong dữ liệu.

**2. Phân tích ma trận nhầm lẫn (Confusion Matrix) Ma trận nhầm lẫn cho biết mức độ mô hình dự đoán đúng hoặc sai từng lớp:**
- True Negative (TN) = 792: dự đoán đúng nấm ăn được.
- False Positive (FP) = 51: dự đoán sai nấm ăn được thành độc.
- False Negative (FN) = 262: dự đoán sai nấm độc thành ăn được.
- True Positive (TP) = 520: dự đoán đúng nấm độc.

Điểm đáng lo ngại nhất là FN = 262: Số lượng FN cao nguy hiểm trong thực tế, vì nấm độc bị dự đoán sai thành ăn được dẫn đến rủi ro sức khoẻ.

→ Có 262 cây nấm độc nhưng mô hình lại dự đoán nhầm là ăn được.

→ Đây là lỗi nghiêm trọng trong bài toán phân loại nấm.

**3. Nhận xét từ Classification Report**

Lớp 0 (ăn được)
- Precision = 0.75 → khá thấp
- Recall = 0.94 → mô hình nhận đúng nhiều nấm ăn được
- F1 = 0.84

Lớp 1 (độc)
- Precision = 0.91 → khi dự đoán là độc thì khá chính xác
- Recall = 0.66 → mô hình bỏ sót rất nhiều nấm độc
- F1 = 0.77

→ Nhìn chung, mô hình thiên lệch về lớp “ăn được” và không nhận diện tốt lớp “độc”. Không phù hợp để sử dụng trong thực tế vì bỏ sót quá nhiều nấm độc, dù độ chính xác nhìn chung không quá tệ.

**4. Nguyên nhân mô hình hoạt động kém**

Dữ liệu mushroom gồm 23 thuộc tính phân loại (categorical) → nhưng bạn đang dùng:
- GaussianNB → dành cho dữ liệu liên tục (continuous)
- hoặc MultinomialNB + LabelEncoder → không phù hợp dữ liệu ký hiệu rời rạc

=> Đây là lý do chính khiến độ chính xác thấp (~80%).

# 6. Tiến hành thử mô hình BernoulliNB

In [6]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


In [7]:
# One-hot toàn bộ dữ liệu
ct = ColumnTransformer(
    [('onehot', OneHotEncoder(), x.columns)], remainder='drop'
)

x_encoded = ct.fit_transform(x)

# Chia lại tập
x_train2, x_test2, y_train2, y_test2 = train_test_split(
    x_encoded, y, test_size=0.2, random_state=42
)

# Mô hình BernoulliNB
model2 = BernoulliNB()
model2.fit(x_train2, y_train2)

# Đánh giá
y_pred2 = model2.predict(x_test2)

acc2 = accuracy_score(y_test2, y_pred2)
cm2 = confusion_matrix(y_test2, y_pred2)
report2 = classification_report(y_test2, y_pred2)

print("Accuracy:", acc2)
print("Confusion Matrix:\n", cm2)
print("Classification Report:\n", report2)

Accuracy: 0.936
Confusion Matrix:
 [[827  16]
 [ 88 694]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.98      0.94       843
           1       0.98      0.89      0.93       782

    accuracy                           0.94      1625
   macro avg       0.94      0.93      0.94      1625
weighted avg       0.94      0.94      0.94      1625



**Nhận xét mô hình BernoulliNB:**

**1. Độ chính xác được cải thiện rõ rệt**
- Độ chính xác (Accuracy) ≈ 93.6% (tăng mạnh so với 80.7%)

**2. Phân tích ma trận nhầm lẫn (Confusion Matrix)**
- Dự đoán đúng nấm ăn được: 827
- Dự đoán đúng nấm độc: 694
- Sai:
  + 16 nấm ăn được bị đoán nhầm là độc (FP)
  + 88 nấm độc bị đoán nhầm là ăn được (FN)

So với mô hình Naive Bayes ban đầu:
- FN giảm mạnh từ 262 → 88
- FP giảm từ 51 → 16

**3. Nhận xét từ Classification Report**

Lớp 0:
- Precision và Recall của 2 lớp đều cao (≈0.90–0.98) → rất tốt, hầu như nhận đúng nấm ăn được.

Lớp 1:
- precision: 0.98 (rất ít nấm “an toàn” bị đoán nhầm là độc)
- recall: 0.89 (nhận diện phần lớn nấm độc, tốt hơn nhiều so với 0.66 trước đó)

F1-score đạt ~0.93 cho cả hai lớp, cân bằng và ổn định.



**Đánh giá chung mô hình BernoulliNB:**
- Mô hình nhận diện nấm độc tốt hơn rất nhiều.
- FN giảm mạnh cho thấy mô hình an toàn hơn, ít bỏ sót nấm độc.
- BernoulliNB phù hợp với dữ liệu dạng phân loại và one-hot, nên cho hiệu quả tốt hơn.

  **So sánh hai mô hình:**
  - Mô hình 2 vượt trội cả về độ chính xác tổng thể và đặc biệt là khả năng nhận diện nấm độc.
  - Với bài toán an toàn (tránh ăn nhầm nấm độc), giảm FN là cực kỳ quan trọng, và BernoulliNB làm tốt việc này hơn.

## 7. Kết luận - Phương án đúng đắn

Dựa trên kết quả thực nghiệm:

→ Mô hình BernoulliNB là phương án tối ưu cho tập dữ liệu mushroom.

Vì:
- Độ chính xác cao hơn đáng kể (93.6% vs 80.7%).
- Khả năng nhận diện nấm độc tốt hơn (Recall 0.89 vs 0.66).
- Ít bỏ sót nấm độc, an toàn hơn khi triển khai thực tế. Giảm FN 262 → 88 là cải tiến quan trọng nhất, vì giảm rủi ro dự đoán sai nấm độc thành ăn được.
- Phù hợp với bản chất dữ liệu categorical.

=> BernoulliNB là lựa chọn đúng đắn và hiệu quả hơn hẳn so với mô hình Naïve Bayes ban đầu, và nên được sử dụng cho bài toán phân loại nấm.

# Kết thúc